# EDA détaillée - Échantillon stratifié Cifer Fraud Detection

À exécuter sur Cifer-echantillon-strat.csv (généré par echantillonnage_strat.py)
 
5 axes d'analyse :
  1. Distribution des types de transactions et lien avec la fraude
  2. Déséquilibre de classes + répartition temporelle des fraudes
  3. Distribution des montants/soldes + patterns de fraude
  4. Corrélations entre variables et avec isFraud
  5. Analyse de isFlaggedFraud vs isFraud
 
Auteur : Rasmané

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)
sns.set_theme(style="whitegrid")
DOSSIER_SORTIE = r"C:\Users\hp\Documents\Fraude_detection\data"

## 0. CHARGEMENT DE L'ÉCHANTILLON STRATIFIÉ

In [ ]:
CHEMIN_FICHIER = r"C:\Users\hp\Documents\Fraude_detection\data\Cifer-echantillon-strat.csv"
 
DTYPES = {
    "step": "int16",
    "type": "category",
    "amount": "float32",
    "oldbalanceOrg": "float32",
    "newbalanceOrig": "float32",
    "oldbalanceDest": "float32",
    "newbalanceDest": "float32",
    "isFraud": "int8",
    "isFlaggedFraud": "int8",
}
 
data1 = pd.read_csv(CHEMIN_FICHIER, dtype=DTYPES)
print(f"Échantillon chargé : {data1.shape[0]:,} lignes, {data1.shape[1]} colonnes")
print(f"Empreinte mémoire : {data1.memory_usage(deep=True).sum() / 1024**2:.1f} Mo")
print(data1.columns.tolist())
 
# Rappel : cet échantillon N'EST PAS le dataset complet.
# Le ratio de déséquilibre ici (~50:1) est différent du ratio réel du
# dataset d'origine (~763:1) -> à toujours préciser dans les commentaires
# et dans le rapport pour ne pas induire en erreur sur la sévérité réelle
# du déséquilibre.
ratio_echantillon = (data1["isFraud"] == 0).sum() / (data1["isFraud"] == 1).sum()
print(f"\nRatio de déséquilibre DANS CET ÉCHANTILLON : {ratio_echantillon:.1f} : 1")
print("(rappel : ratio réel dans le dataset complet ~763.5 : 1, cf. EDA sur fichiers bruts)")

## AXE 1 — DISTRIBUTION DES TYPES DE TRANSACTIONS ET LIEN AVEC LA FRAUDE

In [ ]:
print("\n" + "=" * 70)
print("AXE 1 : TYPES DE TRANSACTIONS ET FRAUDE")
print("=" * 70)
 
repartition_types = data1["type"].value_counts()
repartition_types_pct = data1["type"].value_counts(normalize=True) * 100
tableau_types = pd.DataFrame({
    "nb_transactions": repartition_types,
    "pct_transactions": repartition_types_pct.round(3)
})
print("\n1.1 Répartition globale par type (dans l'échantillon) :")
print(tableau_types)
 
tableau_fraude_type = data1.groupby("type", observed=True)["isFraud"].agg(
    nb_fraudes="sum", nb_transactions="count", taux_fraude="mean"
)
tableau_fraude_type["taux_fraude_pct"] = (tableau_fraude_type["taux_fraude"] * 100).round(4)
tableau_fraude_type = tableau_fraude_type.sort_values("nb_fraudes", ascending=False)
print("\n1.2 Fraude par type de transaction :")
print(tableau_fraude_type)
 
types_avec_fraude = tableau_fraude_type[tableau_fraude_type["nb_fraudes"] > 0].index.tolist()
print(f"\nTypes concernés par au moins une fraude : {types_avec_fraude}")
 
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.barplot(x=repartition_types.index, y=repartition_types.values,
            ax=axes[0], hue=repartition_types.index, palette="Blues_d", legend=False)
axes[0].set_yscale("log")
axes[0].set_title("Volumétrie par type (échantillon, échelle log)")
axes[0].tick_params(axis="x", rotation=45)
 
sns.barplot(x=tableau_fraude_type.index, y=tableau_fraude_type["taux_fraude_pct"],
            ax=axes[1], hue=tableau_fraude_type.index, palette="Reds_d", legend=False)
axes[1].set_title("Taux de fraude par type (%)")
axes[1].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig(f"{DOSSIER_SORTIE}/axe1_types_transactions.png", dpi=120)
#plt.close()
print("Graphique sauvegardé : axe1_types_transactions.png")
 
if types_avec_fraude:
    sous_ensemble = data1[data1["type"].isin(types_avec_fraude)]
    print(f"\n1.4 Volume concerné par les types frauduleux : {len(sous_ensemble):,} "
          f"({len(sous_ensemble)/len(data1)*100:.1f}% de l'échantillon)")
    print(f"Taux de fraude sur ce sous-ensemble : {sous_ensemble['isFraud'].mean()*100:.4f}%")

## AXE 2 — DÉSÉQUILIBRE DE CLASSES + RÉPARTITION TEMPORELLE

In [ ]:
print("\n" + "=" * 70)
print("AXE 2 : DÉSÉQUILIBRE DE CLASSES ET RÉPARTITION TEMPORELLE")
print("=" * 70)
 
effectifs_cible = data1["isFraud"].value_counts()
print(f"\n2.1 Répartition de la cible (échantillon) :")
print(effectifs_cible)
print(f"Pourcentage de fraude dans l'échantillon : {data1['isFraud'].mean()*100:.4f}%")
print(f"Ratio de déséquilibre échantillon : {ratio_echantillon:.1f} : 1")
print("ATTENTION : ce ratio résulte du sous-échantillonnage volontaire de la")
print("non-fraude (RAM limitée à 8 Go) -> ne pas le confondre avec le ratio")
print("réel du dataset complet (~763.5:1), à toujours mentionner ensemble")
print("dans le rapport pour ne pas fausser la lecture de la sévérité réelle.")
 
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
effectifs_cible.plot(kind="bar", ax=axes[0], color=["#2E86AB", "#E63946"])
axes[0].set_title("Répartition des classes (échantillon)")
axes[0].set_xticklabels(["Non-fraude (0)", "Fraude (1)"], rotation=0)
axes[1].pie(effectifs_cible, labels=["Non-fraude", "Fraude"], autopct="%1.2f%%",
            colors=["#2E86AB", "#E63946"])
axes[1].set_title("Répartition des classes (%)")
plt.tight_layout()
plt.savefig(f"{DOSSIER_SORTIE}/axe2_desequilibre_classes.png", dpi=120)
plt.close()
print("Graphique sauvegardé : axe2_desequilibre_classes.png")
 
if "step" in data1.columns:
    print("\n2.3 Répartition temporelle des fraudes (step = 1 heure, 744 steps = 30 jours) :")
    fraudes_par_step = data1[data1["isFraud"] == 1].groupby("step").size()
    taux_par_step = data1.groupby("step")["isFraud"].mean()
 
    print(f"Nombre de steps distincts dans l'échantillon : {data1['step'].nunique()} / 744")
    print(f"Step avec le plus de fraudes en absolu : {fraudes_par_step.idxmax()} "
          f"({fraudes_par_step.max()} fraudes)")
    print(f"Step avec le taux de fraude le plus élevé : {taux_par_step.idxmax()} "
          f"({taux_par_step.max()*100:.2f}%)")
 
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    fraudes_par_step.plot(ax=axes[0], color="#E63946")
    axes[0].set_title("Nombre de fraudes par step (échantillon)")
    axes[0].set_xlabel("step")
    taux_par_step.plot(ax=axes[1], color="#2E86AB")
    axes[1].set_title("Taux de fraude par step (échantillon)")
    axes[1].set_xlabel("step")
    plt.tight_layout()
    plt.savefig(f"{DOSSIER_SORTIE}/axe2_repartition_temporelle.png", dpi=120)
    #plt.close()
    print("Graphique sauvegardé : axe2_repartition_temporelle.png")
 
    # Heure du jour (step % 24) et jour de la semaine (step // 24 % 7)
    data1["heure_du_jour"] = data1["step"] % 24
    data1["jour_simulation"] = data1["step"] // 24
 
    taux_par_heure = data1.groupby("heure_du_jour")["isFraud"].mean()
    print("\nTaux de fraude par heure du jour (step % 24) :")
    print((taux_par_heure * 100).round(4))
 
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    taux_par_heure.plot(kind="bar", ax=axes[0], color="#F4A261")
    axes[0].set_title("Taux de fraude par heure du jour")
    axes[0].set_xlabel("Heure (0-23)")
 
    taux_par_jour = data1.groupby("jour_simulation")["isFraud"].mean()
    taux_par_jour.plot(ax=axes[1], color="#2E86AB", marker="o")
    axes[1].set_title("Taux de fraude par jour de simulation (0-29)")
    axes[1].set_xlabel("Jour")
    plt.tight_layout()
    plt.savefig(f"{DOSSIER_SORTIE}/axe2_taux_par_heure_et_jour.png", dpi=120)
    #plt.close()
    print("Graphique sauvegardé : axe2_taux_par_heure_et_jour.png")
else:
    print("Colonne 'step' absente.")

## AXE 3 — DISTRIBUTION DES MONTANTS/SOLDES + PATTERNS DE FRAUDE

In [ ]:
print("\n" + "=" * 70)
print("AXE 3 : MONTANTS, SOLDES ET PATTERNS DE FRAUDE")
print("=" * 70)
 
colonnes_montant = ["amount", "oldbalanceOrg", "newbalanceOrig",
                     "oldbalanceDest", "newbalanceDest"]
 
print("\n3.1 Statistiques descriptives par classe :")
for col in colonnes_montant:
    comparaison = pd.DataFrame({
        "non_fraude": data1.loc[data1["isFraud"] == 0, col].describe(),
        "fraude": data1.loc[data1["isFraud"] == 1, col].describe()
    })
    print(f"\n--- {col} ---")
    print(comparaison.round(2))
 
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
for i, col in enumerate(colonnes_montant):
    sns.histplot(np.log1p(data1.loc[data1["isFraud"] == 0, col]),
                 color="#2E86AB", label="Non-fraude", alpha=0.5, ax=axes[i], stat="density")
    sns.histplot(np.log1p(data1.loc[data1["isFraud"] == 1, col]),
                 color="#E63946", label="Fraude", alpha=0.5, ax=axes[i], stat="density")
    axes[i].set_title(f"{col} (log1p)")
    axes[i].legend()
for j in range(len(colonnes_montant), len(axes)):
    fig.delaxes(axes[j])
plt.tight_layout()
plt.savefig(f"{DOSSIER_SORTIE}/axe3_distributions_montants.png", dpi=120)
#plt.close()
print("\nGraphique sauvegardé : axe3_distributions_montants.png")
 
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
for i, col in enumerate(colonnes_montant):
    sns.boxplot(data=data1, x="isFraud", y=col, ax=axes[i], hue="isFraud",
                palette=["#2E86AB", "#E63946"], legend=False)
    axes[i].set_yscale("log")
    axes[i].set_title(f"{col} par classe (log)")
    axes[i].set_xticks([0, 1])
    axes[i].set_xticklabels(["Non-fraude", "Fraude"])
for j in range(len(colonnes_montant), len(axes)):
    fig.delaxes(axes[j])
plt.tight_layout()
plt.savefig(f"{DOSSIER_SORTIE}/axe3_boxplots_par_classe.png", dpi=120)
#plt.close()
print("Graphique sauvegardé : axe3_boxplots_par_classe.png")
 
print("\n3.4 Pattern : solde destinataire à 0 avant ET après réception")
masque_solde_fige = (
    (data1["oldbalanceDest"] == 0) & (data1["newbalanceDest"] == 0) & (data1["amount"] > 0)
)
taux_masque = data1.loc[masque_solde_fige, "isFraud"].mean()
taux_hors_masque = data1.loc[~masque_solde_fige, "isFraud"].mean()
print(f"Transactions concernées : {masque_solde_fige.sum():,}")
print(f"Taux de fraude dans ce pattern : {taux_masque*100:.2f}%")
print(f"Taux de fraude hors pattern : {taux_hors_masque*100:.2f}%")
if taux_hors_masque > 0:
    print(f"Ratio : {taux_masque/taux_hors_masque:.1f}x plus de fraude dans ce pattern")
 
print("\n3.5 Pattern : solde émetteur vidé intégralement")
masque_compte_vide = (data1["newbalanceOrig"] == 0) & (data1["oldbalanceOrg"] > 0)
taux_vide = data1.loc[masque_compte_vide, "isFraud"].mean()
taux_non_vide = data1.loc[~masque_compte_vide, "isFraud"].mean()
print(f"Transactions concernées : {masque_compte_vide.sum():,}")
print(f"Taux de fraude (compte vidé) : {taux_vide*100:.2f}%")
print(f"Taux de fraude (compte non vidé) : {taux_non_vide*100:.2f}%")
 
print("\n3.6 Cohérence comptable des soldes émetteur")
data1["erreur_balance_orig"] = (
    data1["oldbalanceOrg"] - data1["amount"] - data1["newbalanceOrig"]
)
incoherences = (data1["erreur_balance_orig"].abs() > 1).sum()
print(f"Transactions avec incohérence de solde : {incoherences:,} "
      f"({incoherences/len(data1)*100:.2f}%)")
corr_erreur = data1[["erreur_balance_orig", "isFraud"]].corr().iloc[0, 1]
print(f"Corrélation erreur_balance_orig / isFraud : {corr_erreur:.4f}")
 
print("\n3.7 Taux de fraude : outliers (IQR) vs valeurs normales")
resultats_outliers = []
for col in colonnes_montant:
    q1, q3 = data1[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    seuil = q3 + 1.5 * iqr
    taux_out = data1.loc[data1[col] > seuil, "isFraud"].mean()
    taux_in = data1.loc[data1[col] <= seuil, "isFraud"].mean()
    resultats_outliers.append({
        "variable": col,
        "nb_outliers": int((data1[col] > seuil).sum()),
        "taux_fraude_outliers_pct": round(taux_out * 100, 3),
        "taux_fraude_normal_pct": round(taux_in * 100, 3),
        "ratio": round(taux_out / taux_in, 2) if taux_in > 0 else None
    })
df_outliers_fraude = pd.DataFrame(resultats_outliers)
print(df_outliers_fraude)
print("-> Ratio > 1 : les outliers concentrent davantage de fraude, ne pas les supprimer.")

## AXE 4 — CORRÉLATIONS ENTRE VARIABLES ET AVEC isFraud

In [ ]:
print("\n" + "=" * 70)
print("AXE 4 : CORRÉLATIONS")
print("=" * 70)
 
colonnes_num = data1.select_dtypes(include=[np.number]).columns.tolist()
matrice_corr = data1[colonnes_num].corr()
 
print("\n4.1 Matrice de corrélation complète :")
print(matrice_corr.round(3))
 
plt.figure(figsize=(12, 10))
sns.heatmap(matrice_corr, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, linewidths=0.5, annot_kws={"size": 8})
plt.title("Matrice de corrélation - variables numériques (échantillon)")
plt.tight_layout()
plt.savefig(f"{DOSSIER_SORTIE}/axe4_matrice_correlation.png", dpi=120)
#plt.close()
print("\nGraphique sauvegardé : axe4_matrice_correlation.png")
 
print("\n4.2 Corrélation de chaque variable avec isFraud (triée) :")
corr_avec_cible = matrice_corr["isFraud"].drop("isFraud").sort_values(ascending=False)
print(corr_avec_cible)
 
plt.figure(figsize=(8, 6))
corr_avec_cible.plot(kind="barh", color=["#E63946" if v > 0 else "#2E86AB" for v in corr_avec_cible])
plt.title("Corrélation avec isFraud (échantillon)")
plt.xlabel("Coefficient de corrélation")
plt.tight_layout()
plt.savefig(f"{DOSSIER_SORTIE}/axe4_correlation_cible.png", dpi=120)
#plt.close()
print("Graphique sauvegardé : axe4_correlation_cible.png")
 
print("\nNote : corrélation de Pearson = relation LINÉAIRE uniquement.")
print("Les patterns non linéaires (axe 3, ex: solde figé à 0) peuvent avoir")
print("un fort pouvoir discriminant malgré une corrélation linéaire faible.")

## AXE 5 — isFlaggedFraud vs isFraud

In [ ]:
print("\n" + "=" * 70)
print("AXE 5 : isFlaggedFraud vs isFraud")
print("=" * 70)
 
if "isFlaggedFraud" in data1.columns:
    print("\n5.1 Répartition de isFlaggedFraud :")
    print(data1["isFlaggedFraud"].value_counts())
 
    print("\n5.2 Table de contingence isFraud x isFlaggedFraud :")
    contingence = pd.crosstab(data1["isFraud"], data1["isFlaggedFraud"],
                               rownames=["isFraud (réel)"], colnames=["isFlaggedFraud (règle)"])
    print(contingence)
 
    vrais_positifs = ((data1["isFraud"] == 1) & (data1["isFlaggedFraud"] == 1)).sum()
    faux_negatifs = ((data1["isFraud"] == 1) & (data1["isFlaggedFraud"] == 0)).sum()
    faux_positifs = ((data1["isFraud"] == 0) & (data1["isFlaggedFraud"] == 1)).sum()
    vrais_negatifs = ((data1["isFraud"] == 0) & (data1["isFlaggedFraud"] == 0)).sum()
 
    total_fraudes = data1["isFraud"].sum()
    rappel = vrais_positifs / total_fraudes if total_fraudes > 0 else 0
    precision = vrais_positifs / (vrais_positifs + faux_positifs) if (vrais_positifs + faux_positifs) > 0 else 0
 
    print(f"\n5.3 Performance de isFlaggedFraud comme détecteur simple :")
    print(f"VP={vrais_positifs}  FN={faux_negatifs}  FP={faux_positifs}  VN={vrais_negatifs}")
    print(f"Rappel : {rappel*100:.2f}% des fraudes détectées par la règle simple")
    print(f"Précision : {precision*100:.2f}% des alertes sont de vraies fraudes")
    print("\n-> Un rappel faible justifie le recours au ML pour ce projet.")
 
    plt.figure(figsize=(6, 5))
    sns.heatmap(contingence, annot=True, fmt="d", cmap="YlOrRd")
    plt.title("isFraud vs isFlaggedFraud (échantillon)")
    plt.tight_layout()
    plt.savefig(f"{DOSSIER_SORTIE}/axe5_isflaggedfraud_vs_isfraud.png", dpi=120)
    #plt.close()
    print("Graphique sauvegardé : axe5_isflaggedfraud_vs_isfraud.png")
else:
    print("Colonne 'isFlaggedFraud' absente.")
 
 
print("\n" + "=" * 70)
print("EDA DÉTAILLÉE TERMINÉE (sur échantillon) — graphiques dans :", DOSSIER_SORTIE)
print("RAPPEL : préciser dans le rapport que cette EDA porte sur un")
print("échantillon stratifié (100% fraudes + sous-échantillon non-fraude,")
print(f"ratio {ratio_echantillon:.0f}:1), et non sur les 21M lignes complètes.")
print("=" * 70)